# 03 — Анализ и сравнение CNN14 vs ResNet18

**Что делает ноутбук** (полный научный анализ):
1. **Test-set оценка**: для каждой best-модели прогон на test-фолде → метрики (accuracy, macro-F1, per-class, confusion matrix).
2. **Aggregated metrics**: mean ± std × bootstrap CI по 15 runs.
3. **Paired Wilcoxon** на per-track правильности.
4. **Training dynamics**: loss/F1 curves с std-band, time-to-90%.
5. **Computational cost**: params, FLOPs, время.
6. **Calibration**: ECE + reliability diagram.
7. **Embeddings**: t-SNE/UMAP penultimate.
8. **Grad-CAM**: side-by-side на одном треке из каждого жанра.
9. **Channel strategy ablation** (для ResNet18).

**Выходы**: `/kaggle/working/figures/` + `final_metrics.csv` + `final_report.json`.

**Время:** ~1 час.

In [ ]:
!pip install -q librosa==0.10.1 h5py soxr fvcore grad-cam umap-learn 2>&1 | tail -3

In [ ]:
import sys
from pathlib import Path
for p in ['/kaggle/input/gtzan-cnn14-resnet18-src', '/kaggle/input/cnn14-resnet18-src',
          '/kaggle/working', str(Path.cwd().parent)]:
    if Path(p, 'src', '__init__.py').exists():
        sys.path.insert(0, p)
        break
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import GENRES, NUM_CLASSES
from src.configs import TrainConfig, AugConfig
from src.dataset import GTZANDataset, load_folds
from src.models import build_model
from src.eval import (
    aggregate_window_predictions, compute_metrics, bootstrap_ci,
    wilcoxon_pair, compute_ece, time_to_fraction_of_best,
)
from src.viz import (
    plot_training_curves, plot_confusion_matrices, plot_per_class_f1,
    plot_embeddings_2d, plot_reliability, compute_gradcam, plot_gradcam_grid,
    plot_cost_vs_accuracy, plot_time_to_fraction,
)
from src.utils import count_parameters, Timer
from torch.utils.data import DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device}')

In [ ]:
# Пути
def find_file(*c):
    for x in c:
        if Path(x).exists():
            return Path(x)
    return None

H5_PATH = find_file('/kaggle/input/gtzan-preproc/gtzan_logmel.h5',
                    '/kaggle/working/gtzan_logmel.h5',
                    Path.cwd().parent / 'outputs' / 'gtzan_logmel.h5')
FOLDS_PATH = find_file('/kaggle/input/gtzan-preproc/folds.json',
                       '/kaggle/working/folds.json',
                       Path.cwd().parent / 'outputs' / 'folds.json')
# Чекпойнты могут лежать в kaggle/input (если прикрепили выводы 01/02 как Dataset) или в working
CKPT_DIRS = [Path('/kaggle/input/gtzan-cnn14-results/outputs'),
             Path('/kaggle/input/gtzan-resnet18-results/outputs'),
             Path('/kaggle/working/outputs'),
             Path.cwd().parent / 'outputs']
OUT_DIR = Path('/kaggle/working/outputs')
FIG_DIR = Path('/kaggle/working/figures')
FIG_DIR.mkdir(exist_ok=True, parents=True)

folds = load_folds(FOLDS_PATH)
print(f'H5: {H5_PATH}')
print(f'Folds: {FOLDS_PATH}')
print(f'Figures: {FIG_DIR}')

## 1. Test-set оценка всех 15 runs на каждой модели

In [ ]:
def find_ckpt(model: str, fold: int, seed: int) -> Path | None:
    name = f'best_{model}_fold{fold}_seed{seed}.pth'
    for d in CKPT_DIRS:
        if (d / name).exists():
            return d / name
    return None

@torch.no_grad()
def evaluate_on_test(model_name: str, fold: int, seed: int,
                     channel_strategy: str = 'avg_conv1'):
    ckpt_path = find_ckpt(model_name, fold, seed)
    if ckpt_path is None:
        return None
    test_ids = folds[str(fold)]['test']
    ds = GTZANDataset(str(H5_PATH), test_ids, mode='eval', aug_cfg=AugConfig(use_specaugment=False, use_mixup=False))
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)

    model = build_model(model_name, channel_strategy=channel_strategy,
                        pann_ckpt_path=None, pretrained=False)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['state_dict'], strict=False)
    model.to(device).eval()

    all_probs, all_labels, all_groups, all_emb = [], [], [], []
    for mel, label, group in loader:
        mel = mel.to(device)
        out = model(mel)
        probs = torch.softmax(out['logits'].float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_emb.append(out['embedding'].cpu().numpy())
        all_labels.extend(label.tolist())
        all_groups.extend(group.tolist())
    w_probs = np.concatenate(all_probs)
    w_emb = np.concatenate(all_emb)
    t_probs, t_labels = aggregate_window_predictions(
        w_probs, np.array(all_labels), np.array(all_groups), 'median')
    # Aggregate embeddings per track (mean)
    groups_arr = np.array(all_groups)
    unique = np.unique(groups_arr)
    t_emb = np.stack([w_emb[groups_arr == g].mean(0) for g in unique])
    t_preds = t_probs.argmax(1)
    metrics = compute_metrics(t_labels, t_preds, t_probs)

    return {'metrics': metrics, 'track_probs': t_probs, 'track_labels': t_labels,
            'track_preds': t_preds, 'track_emb': t_emb}

# Прогон evaluate на всех 15 runs обеих моделей
from src.configs import DEFAULT_SEEDS, N_FOLDS
results = {'cnn14': [], 'resnet18': []}
for model_name in ['cnn14', 'resnet18']:
    for fold in range(N_FOLDS):
        for seed in DEFAULT_SEEDS:
            r = evaluate_on_test(model_name, fold, seed)
            if r is None:
                print(f'  [skip] {model_name} fold{fold} seed{seed}')
                continue
            r['fold'] = fold
            r['seed'] = seed
            r['model'] = model_name
            results[model_name].append(r)
            print(f'{model_name} f{fold}s{seed}: acc={r["metrics"]["accuracy"]:.3f} f1={r["metrics"]["macro_f1"]:.3f}')

## 2. Сводная таблица + bootstrap CI

In [ ]:
import json
summary = []
for model_name in ['cnn14', 'resnet18']:
    res = results[model_name]
    if not res:
        continue
    accs = np.array([r['metrics']['accuracy'] for r in res])
    f1s = np.array([r['metrics']['macro_f1'] for r in res])

    # Bootstrap CI на per-track базе (объединяя все runs)
    all_correct = np.concatenate([(r['track_preds'] == r['track_labels']).astype(int) for r in res])
    mean_acc, ci_low, ci_high = bootstrap_ci(all_correct, n_resamples=1000)
    summary.append({
        'model': model_name,
        'mean_accuracy': float(accs.mean()),
        'std_accuracy': float(accs.std()),
        'mean_macro_f1': float(f1s.mean()),
        'std_macro_f1': float(f1s.std()),
        'bootstrap_mean_acc': mean_acc,
        'bootstrap_ci95_low': ci_low,
        'bootstrap_ci95_high': ci_high,
        'n_runs': len(res),
    })
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(OUT_DIR / 'final_metrics.csv', index=False)

## 3. Paired Wilcoxon на per-track правильности

In [ ]:
# Для каждой пары (fold, seed) собираем correct vectors обеих моделей по тем же track_ids
wilcoxon_pairs = []
for fold in range(N_FOLDS):
    for seed in DEFAULT_SEEDS:
        r_c = next((r for r in results['cnn14'] if r['fold'] == fold and r['seed'] == seed), None)
        r_r = next((r for r in results['resnet18'] if r['fold'] == fold and r['seed'] == seed), None)
        if r_c is None or r_r is None:
            continue
        # Test-set один и тот же → порядок track_labels гарантированно одинаков
        assert (r_c['track_labels'] == r_r['track_labels']).all()
        correct_c = (r_c['track_preds'] == r_c['track_labels']).astype(int)
        correct_r = (r_r['track_preds'] == r_r['track_labels']).astype(int)
        w = wilcoxon_pair(correct_c, correct_r)
        w['fold'] = fold
        w['seed'] = seed
        wilcoxon_pairs.append(w)
wilcoxon_df = pd.DataFrame(wilcoxon_pairs)
print('Wilcoxon (cnn14 vs resnet18) per (fold, seed):')
print(wilcoxon_df[['fold','seed','mean_diff_pp','p_value','n_diff']].to_string(index=False))

# Объединённый тест (concat все per-track) — самая сильная статистика
all_c = np.concatenate([(r['track_preds'] == r['track_labels']).astype(int) for r in results['cnn14']])
all_r = np.concatenate([(r['track_preds'] == r['track_labels']).astype(int) for r in results['resnet18']])
global_w = wilcoxon_pair(all_c, all_r)
print(f'\nGlobal Wilcoxon: mean_diff={global_w["mean_diff_pp"]:.2f} п.п., p={global_w["p_value"]:.4g}')
wilcoxon_df.to_csv(OUT_DIR / 'wilcoxon.csv', index=False)

## 4. Training dynamics curves

In [ ]:
# Загружаем history CSV из обеих моделей
def load_history(model: str) -> pd.DataFrame | None:
    for d in CKPT_DIRS:
        p = d / f'{model}_history.csv'
        if p.exists():
            return pd.read_csv(p)
    return None

hist_c = load_history('cnn14')
hist_r = load_history('resnet18')
if hist_c is not None and hist_r is not None:
    # epoch_global = stage1_epochs + epoch (для stage 2)
    def add_global(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        s1_max = df[df['stage'] == 1].groupby(['fold','seed'])['epoch'].max().max() if (df['stage'] == 1).any() else 0
        df['epoch_global'] = df['epoch'] + np.where(df['stage'] == 2, s1_max + 1, 0)
        # Унифицируем имена
        df = df.rename(columns={'accuracy': 'val_acc', 'macro_f1': 'val_f1'})
        return df
    hist_full = pd.concat([add_global(hist_c), add_global(hist_r)], ignore_index=True)
    fig = plot_training_curves(hist_full, out_path=FIG_DIR / 'fig_4_1_training_curves.png')
    plt.show()
else:
    print('history.csv не найден — пропуск Fig 4.1')

## 5. Confusion matrices + per-class F1

In [ ]:
# Агрегированные CM (суммирование по 15 runs)
cms = {}
per_class_f1 = {}
for model_name in ['cnn14', 'resnet18']:
    if not results[model_name]:
        continue
    cm_sum = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    f1s = []
    for r in results[model_name]:
        cm_sum += np.array(r['metrics']['confusion_matrix'])
        f1s.append(r['metrics']['per_class_f1'])
    cms[model_name] = cm_sum
    per_class_f1[model_name] = np.array(f1s).mean(axis=0).tolist()

plot_confusion_matrices(cms, normalize=True, out_path=FIG_DIR / 'fig_4_4_confusion.png')
plt.show()
plot_per_class_f1(per_class_f1, out_path=FIG_DIR / 'fig_4_3_per_class_f1.png')
plt.show()

## 6. Calibration (ECE + reliability)

In [ ]:
ece_info = {}
eces = {}
for model_name in ['cnn14', 'resnet18']:
    if not results[model_name]:
        continue
    probs = np.concatenate([r['track_probs'] for r in results[model_name]])
    labels = np.concatenate([r['track_labels'] for r in results[model_name]])
    ece, info = compute_ece(labels, probs, n_bins=10)
    ece_info[model_name] = info
    eces[model_name] = ece
    print(f'{model_name}: ECE = {ece:.4f}')
plot_reliability(ece_info, eces, out_path=FIG_DIR / 'fig_4_7_reliability.png')
plt.show()

## 7. t-SNE / UMAP embeddings

In [ ]:
# Берём embeddings с fold=0, seed=42 (один консистентный test split) для каждой модели
emb_data = {}
lbls = None
for model_name in ['cnn14', 'resnet18']:
    r = next((r for r in results[model_name] if r['fold'] == 0 and r['seed'] == 42), None)
    if r is None:
        continue
    emb_data[model_name] = r['track_emb']
    if lbls is None:
        lbls = r['track_labels']
if emb_data:
    plot_embeddings_2d(emb_data, lbls, method='umap',
                       out_path=FIG_DIR / 'fig_4_5_embeddings.png')
    plt.show()

## 8. Grad-CAM side-by-side (один трек на жанр)

In [ ]:
import h5py
# Возьмём fold=0, seed=42 модели обеих архитектур
ck_c = find_ckpt('cnn14', 0, 42)
ck_r = find_ckpt('resnet18', 0, 42)
if ck_c and ck_r:
    m_c = build_model('cnn14', pann_ckpt_path=None)
    m_c.load_state_dict(torch.load(ck_c, map_location='cpu')['state_dict'], strict=False)
    m_c.to(device).eval()
    m_r = build_model('resnet18', channel_strategy='avg_conv1', pretrained=False)
    m_r.load_state_dict(torch.load(ck_r, map_location='cpu')['state_dict'], strict=False)
    m_r.to(device).eval()

    target_c = m_c.backbone.conv_block6
    target_r = m_r.backbone.layer4[1].conv2

    test_ids = folds['0']['test']
    cam_dir = FIG_DIR / 'gradcam'
    cam_dir.mkdir(exist_ok=True)

    with h5py.File(H5_PATH, 'r') as h5:
        for genre in GENRES:
            tid = next((t for t in test_ids if t.startswith(genre)), None)
            if tid is None:
                continue
            mel = h5[tid][:]
            # Возьмём центральное 3-сек окно
            from src.configs import WINDOW_FRAMES
            T = mel.shape[1]
            start = max(0, (T - WINDOW_FRAMES) // 2)
            window = mel[:, start:start + WINDOW_FRAMES]
            x = torch.from_numpy(window).unsqueeze(0).unsqueeze(0).float().to(device)
            # Per-window normalize
            x = (x - x.mean()) / x.std().clamp(min=1e-6)
            x = x.clamp(-5, 5) / 5.0
            from src import GENRE_TO_IDX
            target_cls = GENRE_TO_IDX[genre]
            cam_c = compute_gradcam(m_c, target_c, x.requires_grad_(True), target_cls)
            cam_r = compute_gradcam(m_r, target_r, x.clone().requires_grad_(True), target_cls)
            plot_gradcam_grid(
                mels={'cnn14': window, 'resnet18': window},
                cams={'cnn14': cam_c, 'resnet18': cam_r},
                title=f'Grad-CAM: {genre} ({tid})',
                out_path=cam_dir / f'gradcam_{genre}.png',
            )
            plt.show()

## 9. Computational cost: params, FLOPs, latency

In [ ]:
cost_rows = []
dummy = torch.randn(1, 1, 64, 300).to(device)

for name in ['cnn14', 'resnet18']:
    model = build_model(name, pann_ckpt_path=None, pretrained=False).to(device).eval()
    params_total = count_parameters(model)
    # trainable in stage 2
    if name == 'cnn14':
        model.unfreeze_last_block()
    else:
        model.unfreeze_last_block()
    params_trainable = count_parameters(model, trainable_only=True)
    # FLOPs (только если fvcore доступен)
    try:
        from fvcore.nn import FlopCountAnalysis
        with torch.no_grad():
            flops = FlopCountAnalysis(model, dummy).total()
    except Exception as e:
        flops = -1
        print(f'fvcore err {name}: {e}')
    # latency
    for _ in range(3):  # warmup
        _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    with Timer() as t:
        for _ in range(20):
            _ = model(dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    latency_ms = t.elapsed / 20 * 1000

    cost_rows.append({
        'model': name,
        'params_total_M': params_total / 1e6,
        'params_trainable_M': params_trainable / 1e6,
        'flops_G': flops / 1e9 if flops > 0 else None,
        'latency_ms_per_window': latency_ms,
    })
    del model
    torch.cuda.empty_cache()
cost_df = pd.DataFrame(cost_rows)
print(cost_df.to_string(index=False))
cost_df.to_csv(OUT_DIR / 'computational_cost.csv', index=False)

## 10. Time-to-90% of best val-F1

In [ ]:
ttf_data = {'cnn14': [], 'resnet18': []}
for model_name, hist in [('cnn14', hist_c), ('resnet18', hist_r)]:
    if hist is None:
        continue
    f1_col = 'macro_f1' if 'macro_f1' in hist.columns else 'val_f1'
    for (fold, seed), g in hist.groupby(['fold','seed']):
        # Берём только stage 2 для сравнимости
        g2 = g[g['stage'] == 2].sort_values('epoch')
        if g2.empty:
            continue
        ep = time_to_fraction_of_best(g2[f1_col].tolist(), fraction=0.9)
        if ep is not None:
            ttf_data[model_name].append(ep)
plot_time_to_fraction(ttf_data, fraction=0.9,
                      out_path=FIG_DIR / 'fig_4_9_time_to_90.png')
plt.show()

## 11. ResNet18 channel-strategy ablation

In [ ]:
abl_path = None
for d in CKPT_DIRS:
    if (d / 'resnet18_ablation.csv').exists():
        abl_path = d / 'resnet18_ablation.csv'
        break
if abl_path:
    abl = pd.read_csv(abl_path)
    print('ResNet18 channel-strategy ablation:')
    print(abl.to_string(index=False))
else:
    print('ablation csv не найден')

## 12. Финальный отчёт JSON

In [ ]:
report = {
    'final_metrics': summary_df.to_dict(orient='records') if not summary_df.empty else [],
    'wilcoxon_global': global_w if 'global_w' in dir() else None,
    'ece': eces,
    'computational_cost': cost_df.to_dict(orient='records'),
    'time_to_90_pct_mean': {k: float(np.mean(v)) if v else None for k, v in ttf_data.items()},
}
with open(OUT_DIR / 'final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print('final_report.json сохранён')
print(json.dumps(report, indent=2, ensure_ascii=False, default=str)[:2000])